In [6]:
import os
import glob
import re


In [2]:
input_folder = r"C:/Users/KayzyM4nn/Documents/GitHub/Solution-for-Industry-Knowledge-Hub-Platform/ocr-service/Kayzy"
txt_files = glob.glob(os.path.join(input_folder, "*.[tT][xX][tT]"))
print(f"พบไฟล์ข้อความทั้งหมด {len(txt_files)} ไฟล์")

พบไฟล์ข้อความทั้งหมด 306 ไฟล์


In [30]:
import re
import glob
import os

def process_and_chunk_docs(file_paths, max_chunk_size=1000):
    full_content = ""
    # 1. เรียงลำดับไฟล์ตามตัวเลข (8, 9, 10...) เพื่อความต่อเนื่องของเนื้อหา
    file_paths.sort(key=lambda f: int(re.sub(r'\D', '', os.path.basename(f))))
    print(f"เรียงลำดับไฟล์ใหม่: {[os.path.basename(f) for f in file_paths]}")

    # 2. อ่านและรวมเนื้อหาทั้งหมดเป็นก้อนเดียว
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            full_content += f.read() + "\n\n"

    # 3. Clean ข้อมูลเบื้องต้น
    full_content = re.sub(r'<page_number>.*?</page_number>', '', full_content, flags=re.DOTALL)
    full_content = re.sub(r"\', '", '', full_content)
    full_content = re.sub(r' +', ' ', full_content)

    # 4. แยกเนื้อหาตาม Header (# หรือ ##)
    sections = re.split(r'(^#+ .*|\n#+ .*)', full_content, flags=re.MULTILINE)
    
    chunks = []
    current_header = "บทนำ/General"
    
    for section in sections:
        clean_section = section.strip()
        if not clean_section: continue
        
        # เก็บหัวข้อปัจจุบันเพื่อใช้ฉีด (Inject) เข้าไปในทุก Chunk
        if clean_section.startswith('#'):
            current_header = clean_section
        else:
            # --- กลยุทธ์สะสมเนื้อหา (Accumulation Strategy) ---
            paragraphs = clean_section.split('\n\n')
            buffer_text = ""
            
            for p in paragraphs:
                p = p.strip()
                if not p: continue
                
                # เงื่อนไขพิเศษ: ถ้าเป็น Bullet หรือย่อหน้าก่อนหน้าลงท้ายด้วย : ให้รวมกันทันที
                if (p.startswith(('*', '-')) or buffer_text.endswith(':')):
                    buffer_text += "\n" + p
                # ถ้าความยาวรวมย่อหน้าใหม่ยังไม่เกินกำหนด ให้สะสมต่อไป
                elif len(buffer_text) + len(p) < max_chunk_size:
                    buffer_text = (buffer_text + "\n\n" + p) if buffer_text else p
                else:
                    # ถ้าเกินแล้ว ให้บันทึก Chunk ปัจจุบัน และเริ่ม Buffer ใหม่
                    if buffer_text:
                        chunks.append(f"หัวข้อ: {current_header}\nเนื้อหา: {buffer_text}")
                    buffer_text = p
            
            # เก็บตกเนื้อหาส่วนสุดท้ายใน Section
            if buffer_text:
                chunks.append(f"หัวข้อ: {current_header}\nเนื้อหา: {buffer_text}")

    return chunks

chunks = process_and_chunk_docs(txt_files, max_chunk_size=1000)
print(f"จำนวน Chunk ที่ได้: {len(chunks)}")

# แสดงตัวอย่าง Chunk แรก
for i, chunk in enumerate(chunks[:5]):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

เรียงลำดับไฟล์ใหม่: ['page_8.txt', 'page_9.txt', 'page_10.txt', 'page_11.txt', 'page_12.txt', 'page_13.txt', 'page_14.txt', 'page_15.txt', 'page_16.txt', 'page_17.txt']
จำนวน Chunk ที่ได้: 34
--- Chunk 1 ---
หัวข้อ: ## บทนำของบทเรียน
เนื้อหา: บทเรียนนี้ครอบคลุมแนวคิดพื้นฐานและคำศัพท์ที่สำคัญเกี่ยวกับระบบคลาวด์ โดยมีวัตถุประสงค์ เพื่อให้ผู้ที่เกี่ยวข้องกับการบริหารจัดการระบบคลาวด์สามารถสื่อสารได้อย่างชัดเจนและมีประสิทธิภาพ ภายในเนื้อหาจะมีการให้คำจำกัดความของรูปแบบการให้บริการคลาวด์ต่างๆ ตลอดจนแนวคิดเรื่องโมเดล ความรับผิดชอบร่วมกัน (Shared Responsibility Model) นอกจากนี้ ยังกล่าวถึงแนวโน้มเทคโนโลยีคลาวด์ ที่กำลังพัฒนา และนำเสนอแนวทางการวิเคราะห์และแก้ไขปัญหา (CompTIA Troubleshooting Methodology) ซึ่งเป็นเครื่องมือสำคัญสำหรับผู้ดูแลระบบคลาวด์

--- Chunk 2 ---
หัวข้อ: ## วัตถุประสงค์ของบทเรียน
เนื้อหา: เมื่อจบการเรียนรู้ในบทเรียนนี้ ผู้เรียนจะสามารถ:
* เข้าใจแนวคิดพื้นฐานของระบบคลาวด์
* ตระหนักถึงการพัฒนาและวิวัฒนาการของเทคโนโลยีคลาวด์
* เข้าใจแนวทางการวิเคราะห์และแก้ไขปัญหาอย่างเป็นระบบ 

In [36]:
import re
import glob
import os
from sentence_transformers import SentenceTransformer
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()
db_key = os.getenv("DB_KEY")

def process_to_mongodb(file_paths, mongo_uri, db_name, collection_name):
    full_content = ""
    file_paths.sort(key=lambda f: int(re.sub(r'\D', '', os.path.basename(f))))
    
    # 1. สร้างดัชนีช่วงตัวอักษรของแต่ละหน้า เพื่อความแม่นยำในการระบุเลขหน้า
    page_ranges = []
    current_length = 0
    
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
            # Clean เบื้องต้นในแต่ละไฟล์
            content = re.sub(r'<page_number>.*?</page_number>', '', content, flags=re.DOTALL)
            content = re.sub(r' +', ' ', content)
            
            start_idx = current_length
            full_content += content + "\n\n"
            current_length = len(full_content)
            end_idx = current_length
            
            page_num = int(re.sub(r'\D', '', os.path.basename(path)))
            page_ranges.append({"page": page_num, "start": start_idx, "end": end_idx})

    # 2. แยก Section ตาม Header
    sections = []
    header_pattern = r'(^#+ .*|\n#+ .*)'
    last_idx = 0
    current_header = "บทนำ/General"
    
    for match in re.finditer(header_pattern, full_content, flags=re.MULTILINE):
        section_text = full_content[last_idx:match.start()].strip()
        if section_text:
            sections.append({
                "header": current_header, 
                "text": section_text, 
                "start": last_idx, 
                "end": match.start()
            })
        current_header = match.group().strip()
        last_idx = match.start()
    
    sections.append({
        "header": current_header, 
        "text": full_content[last_idx:].strip(), 
        "start": last_idx, 
        "end": len(full_content)
    })

    # 3. สร้าง Chunks
    max_chunk_size = 1000
    final_raw_chunks = []

    for sec in sections:
        paragraphs = sec['text'].split('\n\n')
        buffer_text = ""
        buffer_start = sec['start']
        
        for p in paragraphs:
            p = p.strip()
            if not p: continue
            
            if (p.startswith(('*', '-')) or buffer_text.endswith(':')):
                buffer_text += "\n" + p
            elif len(buffer_text) + len(p) < max_chunk_size:
                buffer_text = (buffer_text + "\n\n" + p) if buffer_text else p
            else:
                if buffer_text:
                    final_raw_chunks.append({
                        "header": sec['header'], 
                        "text": buffer_text,
                        "start_idx": buffer_start,
                        "end_idx": buffer_start + len(buffer_text)
                    })
                buffer_start = full_content.find(p, buffer_start)
                buffer_text = p
        
        if buffer_text:
            final_raw_chunks.append({
                "header": sec['header'], 
                "text": buffer_text,
                "start_idx": buffer_start,
                "end_idx": buffer_start + len(buffer_text)
            })

    # --- ส่วนประมวลผล Embedding และบันทึกข้อมูล ---
    print("กำลังโหลดโมเดล BAAI/bge-m3...")
    model = SentenceTransformer('BAAI/bge-m3')
    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]
    
    final_documents = []
    print(f"กำลังประมวลผล {len(final_raw_chunks)} chunks...")

    for index, item in enumerate(final_raw_chunks):
        # สร้างข้อความพื้นฐาน
        raw_display_text = f"หัวข้อ: {item['header']}\nเนื้อหา: {item['text']}"
        
        # --- [แก้ไขส่วนที่ feed เข้า DB] ลบสัญลักษณ์และคำที่ไม่ต้องการออกทั้งหมด ---
        # 1. ลบ "หัวข้อ:" และ "เนื้อหา:"
        clean_text = raw_display_text.replace("หัวข้อ:", "").replace("เนื้อหา:", "")
        # 2. ลบเครื่องหมาย Header (#) ทั้งหมด
        clean_text = re.sub(r'#+', '', clean_text)
        # 3. ลบการขึ้นบรรทัดใหม่ (\n) และแทนที่ด้วยช่องว่าง
        clean_text = clean_text.replace('\n', ' ')
        # 4. ลบช่องว่างส่วนเกิน
        clean_text = re.sub(r' +', ' ', clean_text).strip()
        
        # ทำ Embedding จากข้อความที่ผ่านการล้างข้อมูลแล้ว
        embedding = model.encode(clean_text).tolist()

        # ตรวจสอบเลขหน้าที่ถูกต้องจาก Index ช่วงที่บันทึกไว้
        chunk_pages = []
        for pr in page_ranges:
            if not (item['end_idx'] <= pr['start'] or item['start_idx'] >= pr['end']):
                chunk_pages.append(pr['page'])

        # เตรียมข้อมูลสำหรับ Feed เข้า Database
        doc = {
            "chunk_id": index,
            "text": clean_text,              # บันทึกแบบที่ไม่มี หัวข้อ:, เนื้อหา:, \n และ #
            "header": item['header'].replace('#', '').strip(), # บันทึก Header แบบสะอาด
            "page": sorted(list(set(chunk_pages))),
            "embedding": embedding           # Vector จากข้อความที่คลีนแล้ว
        }
        final_documents.append(doc)

    if final_documents:
        collection.delete_many({}) 
        collection.insert_many(final_documents)
        print(f"Feed data เรียบร้อย! จำนวน {len(final_documents)} รายการ")
    
    client.close()

if __name__ == "__main__":
    input_folder = r"C:/Users/KayzyM4nn/Documents/GitHub/Solution-for-Industry-Knowledge-Hub-Platform/ocr-service/Kayzy"
    txt_files = glob.glob(os.path.join(input_folder, "*.[tT][xX][tT]"))
    
    MONGO_URI = db_key
    DB_NAME = "Knowledge_hub"
    COLL_NAME = "Data_project"

    process_to_mongodb(txt_files, MONGO_URI, DB_NAME, COLL_NAME)

กำลังโหลดโมเดล BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 648.27it/s, Materializing param=pooler.dense.weight]                               


กำลังประมวลผล 892 chunks...
Feed data เรียบร้อย! จำนวน 892 รายการ
